<a href="https://colab.research.google.com/github/mnleggette/genomics-bridge-fastq-qc/blob/main/IAN630_Lab02_Data_Quality_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STEP 1, LOAD THE .CSV FILE.

In [5]:
import pandas as pd

df = pd.read_csv("sample_vitals_messy.csv")
df.head()

,patient_id,recorded_at,metric,value,unit,source
0,P001,2026-09-01T08:00:00,heart_rate,72.0,bpm,ehr
1,P001,2026-09-01T08:05:00,heart_rate,740.0,bpm,ehr
2,P001,2026-09-01 08:10,heart_rate,68.0,BPM,watch
3,P001,NaN,heart_rate,70.0,bpm,ehr
4,P002,2026-09-01T09:00:00,heart_rate,NaN,bpm,ehr


In [6]:
print(df.shape)
print(df.isna().sum())
print(df.duplicated().sum())

(14, 6)
patient_id     0
recorded_at    1
metric         0
value          1
unit           0
source         0
dtype: int64
1


NOW, IT'S TIME TO INVESTIGATE OUR DATA. LETS LOOK FOR THE NUMBER OF ROWS AND COLUMNS, MISSING VALUES, AND DUPLICATE ROWS.

In [7]:
print(df.shape)
print(df.isna().sum())
print(df.duplicated().sum())

(14, 6)
patient_id     0
recorded_at    1
metric         0
value          1
unit           0
source         0
dtype: int64
1


LET'S CHECK EVERY ROW...RECORDING ITS QUALITY DIMENSION. SPECIFICALLY ITS COMPLETENESS, CORRECTNESS, PLAUSIBILITY. EVEN IF WE SHOULD FIX, DROP, FLAG, OR ESCALATE IT...

In [8]:
print(df.to_string(index=True))

   patient_id          recorded_at      metric  value  unit    source
0        P001  2026-09-01T08:00:00  heart_rate   72.0   bpm       ehr
1        P001  2026-09-01T08:05:00  heart_rate  740.0   bpm       ehr
2        P001     2026-09-01 08:10  heart_rate   68.0   BPM     watch
3        P001                  NaN  heart_rate   70.0   bpm       ehr
4        P002  2026-09-01T09:00:00  heart_rate    NaN   bpm       ehr
5        P002  2026-09-01T09:00:00         sbp  118.0  mmHg       ehr
6        P002  2026-09-01T09:00:00         sbp  118.0  mmhg    manual
7        P003  2026-09-01T10:00:00  heart_rate   -5.0   bpm     watch
8        P003  2026-09-01T10:00:00  heart_rate   58.0   bpm     watch
9        P003  2026-13-01T10:00:00  heart_rate   60.0   bpm       ehr
10       P004  2026-09-01T11:00:00      temp_c   98.6     C       ehr
11       P004  2026-09-01T11:05:00      temp_c   37.0     C       ehr
12       P005  2026-09-01T12:00:00  heart_rate   88.0   bpm  team_emr
13       P005  2026-

NOW, LET'S LOOK FOR UNUSUAL HEART RATES, MISSING VALUES, INCONSISTENT UNITS, DUPLICATE RECORDS, AND DATES THAT DON'T MAKE SENSE.

In [9]:
display(df)

,patient_id,recorded_at,metric,value,unit,source
0,P001,2026-09-01T08:00:00,heart_rate,72.0,bpm,ehr
1,P001,2026-09-01T08:05:00,heart_rate,740.0,bpm,ehr
2,P001,2026-09-01 08:10,heart_rate,68.0,BPM,watch
3,P001,NaN,heart_rate,70.0,bpm,ehr
4,P002,2026-09-01T09:00:00,heart_rate,NaN,bpm,ehr
5,P002,2026-09-01T09:00:00,sbp,118.0,mmHg,ehr
6,P002,2026-09-01T09:00:00,sbp,118.0,mmhg,manual
7,P003,2026-09-01T10:00:00,heart_rate,-5.0,bpm,watch
8,P003,2026-09-01T10:00:00,heart_rate,58.0,bpm,watch
9,P003,2026-13-01T10:00:00,heart_rate,60.0,bpm,ehr


STEP 2- LIST AT LEAST 6 DISTICT QUALITY ISSUES, BE SPECIFIC: FOW-LEVEL WHEN POSSIBLE.

CREATING THE DATA QUALITY INVESTIGATION TABLE

In [11]:
issues = pd.DataFrame(columns=[
    "Row",
    "Quality issue",
    "Dimension",
    "Proposed handling"
])

display(issues)

,Row,Quality issue,Dimension,Proposed handling


WE HAVE OUR BLANK TABLE< NOW LET'S INVESTIGATE OUR DATASET...

In [12]:
display(df)

,patient_id,recorded_at,metric,value,unit,source
0,P001,2026-09-01T08:00:00,heart_rate,72.0,bpm,ehr
1,P001,2026-09-01T08:05:00,heart_rate,740.0,bpm,ehr
2,P001,2026-09-01 08:10,heart_rate,68.0,BPM,watch
3,P001,NaN,heart_rate,70.0,bpm,ehr
4,P002,2026-09-01T09:00:00,heart_rate,NaN,bpm,ehr
5,P002,2026-09-01T09:00:00,sbp,118.0,mmHg,ehr
6,P002,2026-09-01T09:00:00,sbp,118.0,mmhg,manual
7,P003,2026-09-01T10:00:00,heart_rate,-5.0,bpm,watch
8,P003,2026-09-01T10:00:00,heart_rate,58.0,bpm,watch
9,P003,2026-13-01T10:00:00,heart_rate,60.0,bpm,ehr


SOME CURIOUS THINGS ARE HAPPENING, NOW I ASK MYSELF THE FOLLOWING FOR EACH SUSPICIOUS VALUE...
- ROW: WHERE IS THE PROBLEM?
- ISSUE: SPECIFY THE ROOT CAUSE
- DIMENSION: COMPLETENESS, CORRECTNESS, PLAUSIBILITY, CONCORDANCE, CURRENCY, PROVENANCE?
- HANDLIN': FIX, DROP, FLAG, OR ESCALATE?

NEW NEW VOCABULARY:
*   NaN = NOT A NUMBER
*   sbp = SYSTOLIC BLOOD PRESSURE


In [13]:
display(df.loc[[3, 5, 6]])

,patient_id,recorded_at,metric,value,unit,source
3,P001,NaN,heart_rate,70.0,bpm,ehr
5,P002,2026-09-01T09:00:00,sbp,118.0,mmHg,ehr
6,P002,2026-09-01T09:00:00,sbp,118.0,mmhg,manual


In [14]:
issues.loc[len(issues)] = [
    3,
    "Missing recorded_at timestamp",
    "Completeness",
    "Flag for review"
]

display(issues)

,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review


NOW, I AM CURIOUS IF ROW 3 TRULY HAS A MISSING TIME STAMP SINCE THE RECORDED TIME STAMP SAYS IT IS NOT A NUMBER.

In [15]:
pd.isna(df.loc[3, "recorded_at"])

True

In [16]:
issues.loc[len(issues)] = [
    3,
    "Missing recorded_at timestamp",
    "Completeness",
    "Flag for review"
]

display(issues)

,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review
1,3,Missing recorded_at timestamp,Completeness,Flag for review


I HAVE IDENTIFIED DATA QUALITY **ISSUE #1**.
NEXT, I WILL INVESTIGATE ROWS 5 AND 6 FOR POSSIBLE BLOOD-PRESSURE INCONSISTENCIES.

I HAVE ALSO SUCCESSFULLY DOCUMENTED ISSUE 1: MISSING TIMESTAMP IN ROW 3 IN YM DATA-QUALITY TABLE.

NOW, I WILL INVESTIGATE ISSUE 2...

In [18]:
display(df)

,patient_id,recorded_at,metric,value,unit,source
0,P001,2026-09-01T08:00:00,heart_rate,72.0,bpm,ehr
1,P001,2026-09-01T08:05:00,heart_rate,740.0,bpm,ehr
2,P001,2026-09-01 08:10,heart_rate,68.0,BPM,watch
3,P001,NaN,heart_rate,70.0,bpm,ehr
4,P002,2026-09-01T09:00:00,heart_rate,NaN,bpm,ehr
5,P002,2026-09-01T09:00:00,sbp,118.0,mmHg,ehr
6,P002,2026-09-01T09:00:00,sbp,118.0,mmhg,manual
7,P003,2026-09-01T10:00:00,heart_rate,-5.0,bpm,watch
8,P003,2026-09-01T10:00:00,heart_rate,58.0,bpm,watch
9,P003,2026-13-01T10:00:00,heart_rate,60.0,bpm,ehr


FIELD FINDINGS:

R of -5.0 IS PHYSIOLOGICALLY IMPOSSIBLE

*  HR OF -5.0 IS PHYSIOLOGIALLY IMPOSSIBLE.
*  QUALITY ISSUE: NEGATIVE HR	(-5.0 bpm)
*  DIMENSION: 	PLAUSBILITY
*  HANDLING:	FLAG AND EXCLUDE FROM HR CALCULATIONS.

FIRST, FIND ITS ROW NUMBER.

In [19]:
df[df["value"] == -5.0]

,patient_id,recorded_at,metric,value,unit,source
7,P003,2026-09-01T10:00:00,heart_rate,-5.0,bpm,watch


ADD THE ISSUE THEN REPLACE ROW_NUMBER WITH THE INDEX THAT I FIND...

In [22]:
issues.loc[len(issues)] = [
    7,
    "Impossible heart rate: -5.0 bpm",
    "Plausibility",
    "Flag and exclude from HR summary"
]

display(issues)

,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review
1,3,Missing recorded_at timestamp,Completeness,Flag for review
2,3,Missing recorded_at timestamp,Completeness,Flag for review
3,7,Impossible heart rate: -5.0 bpm,Plausibility,Flag and exclude from HR summary


NOT DEFINED, THEN WE DEFINED IT. AND WE HAVE HANDLED ROW 3 ALREADY.

In [23]:
display(df[df.duplicated(keep=False)])

,patient_id,recorded_at,metric,value,unit,source
12,P005,2026-09-01T12:00:00,heart_rate,88.0,bpm,team_emr
13,P005,2026-09-01T12:00:00,heart_rate,88.0,bpm,team_emr


WE HAVE IDENTIFIED **ISSUE #3** WHICH IS DUPLICATE RECORDS. IN ROWS 12 AND 13 THERE IS A DUPLICATE HEART RATE OF 88. THIS IS VERY PLAUSIBLE AND NOT IMPOSSIBLE, WE JUST NEED TO RECORD THE DUPLICITY...

In [24]:
issues.loc[len(issues)] = [
    "12 and 13",
    "Duplicate record: value 88",
    "Concordance",
    "Flag; remove one copy after verification"
]

display(issues)

,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review
1,3,Missing recorded_at timestamp,Completeness,Flag for review
2,3,Missing recorded_at timestamp,Completeness,Flag for review
3,7,Impossible heart rate: -5.0 bpm,Plausibility,Flag and exclude from HR summary
4,12 and 13,Duplicate record: value 88,Concordance,Flag; remove one copy after verification


MY PROGRESS: 3 of 6 ISSUES
1. ROW 3 — MISSING TIMESTAMP (COMPLETENESS)
2. ROW 7 — PHYSIOLOGICALLY IMPOSSIBLE HR, -5.0 bpm (PLAUSIBILITY)
3. ROWS 12–13 — DUPLICATE RECORD (CONCORDANCE)



NOW TO CHECK FOR INCONSISTENT UNITS OF MEASURE...

In [25]:
display(df[["metric", "unit"]].drop_duplicates())

,metric,unit
0,heart_rate,bpm
2,heart_rate,BPM
5,sbp,mmHg
6,sbp,mmhg
10,temp_c,C


WE HAVE FOUND **ISSUE #4:** UNIT FORMAT INCONSISTENCY

*   ROW:	0 METRIC:	heart_rate UNIT:	bpm PROBLEM: Standard
*   ROW: 2  METRIC: BPM        UNIT: BPM PROBLEM: CAPITALIZATION

*   ROW 5: METRIC: sbp	        UNIT: mmHg	PROBLEM: Standard
*   ROW 6: METRIC: sbp	        UNIT: mmhg	PROBLEM: Inconsistent capitalization






In [26]:
issues.loc[len(issues)] = [
    "2 and 6",
    "Inconsistent unit capitalization: BPM/bpm and mmhg/mmHg",
    "Correctness",
    "Standardize unit labels after verification"
]

display(issues)

,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review
1,3,Missing recorded_at timestamp,Completeness,Flag for review
2,3,Missing recorded_at timestamp,Completeness,Flag for review
3,7,Impossible heart rate: -5.0 bpm,Plausibility,Flag and exclude from HR summary
4,12 and 13,Duplicate record: value 88,Concordance,Flag; remove one copy after verification
5,2 and 6,Inconsistent unit capitalization: BPM/bpm and ...,Correctness,Standardize unit labels after verification


In [27]:
issues = issues.drop_duplicates(
    subset=["Row", "Quality issue"]
).reset_index(drop=True)

display(issues)

,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review
1,7,Impossible heart rate: -5.0 bpm,Plausibility,Flag and exclude from HR summary
2,12 and 13,Duplicate record: value 88,Concordance,Flag; remove one copy after verification
3,2 and 6,Inconsistent unit capitalization: BPM/bpm and ...,Correctness,Standardize unit labels after verification


In [28]:
issues = issues.drop_duplicates(
    subset=["Row", "Quality issue"]
).reset_index(drop=True)

display(issues)

,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review
1,7,Impossible heart rate: -5.0 bpm,Plausibility,Flag and exclude from HR summary
2,12 and 13,Duplicate record: value 88,Concordance,Flag; remove one copy after verification
3,2 and 6,Inconsistent unit capitalization: BPM/bpm and ...,Correctness,Standardize unit labels after verification


MISSING TIMESTAMP RECORDED 3 TIMES IN QUALITY-TABLE, SO WE JUST REMOVED THAT BY ADDING A RECORD TO OUR INVESTIGATION TABLE.

NOW, WE WILL CHECK FOR INVALID DATES...

In [29]:
dates = pd.to_datetime(df["recorded_at"], errors="coerce", format="mixed")

display(df[dates.isna()])

,patient_id,recorded_at,metric,value,unit,source
3,P001,NaN,heart_rate,70.0,bpm,ehr
9,P003,2026-13-01T10:00:00,heart_rate,60.0,bpm,ehr


In [30]:
display(df[
    dates.isna() & df["recorded_at"].notna()
])

,patient_id,recorded_at,metric,value,unit,source
9,P003,2026-13-01T10:00:00,heart_rate,60.0,bpm,ehr


**# ISSUE 5:** TEMPERATURE INCONSISTENCIES...

In [31]:
display(df[df["metric"].str.contains("temp", case=False, na=False)])

,patient_id,recorded_at,metric,value,unit,source
10,P004,2026-09-01T11:00:00,temp_c,98.6,C,ehr
11,P004,2026-09-01T11:05:00,temp_c,37.0,C,ehr


**ISSUE #5:** TEMPERATURE SCALE CONFUSION

* FIELD FINDING
* PROBLEM: TEMPERATURE OF 98.6 LABELED °C
* DIMENSION: PLAUSIBILITY/ CORRECTNESS
* HANDLING: FLAG AND VERIFY THE INTENDED UNIT
* REASON:	98.6°C IS INCOMPATIBLE WITH A LIVING PATIENT'S BODY TEMPERATURE

In [32]:
display(df[(df["metric"].str.contains("temp", case=False, na=False)) & (df["value"] == 98.6)])

,patient_id,recorded_at,metric,value,unit,source
10,P004,2026-09-01T11:00:00,temp_c,98.6,C,ehr


WE HAVE IDENTIFIED THE ROW WHERE PROBLEM LIES...

In [36]:

row = df[
    (df["metric"].str.contains("temp", case=False, na=False)) &
    (df["value"] == 98.6)
].index[0]

issues.loc[len(issues)] = [
    row,
    "98.6 recorded as Celsius; possible Fahrenheit confusion",
    "Plausibility",
    "Flag and verify the original unit"
]

display(issues)


,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review
1,7,Impossible heart rate: -5.0 bpm,Plausibility,Flag and exclude from HR summary
2,12 and 13,Duplicate record: value 88,Concordance,Flag; remove one copy after verification
3,2 and 6,Inconsistent unit capitalization: BPM/bpm and ...,Correctness,Standardize unit labels after verification
4,10,98.6 recorded as Celsius; possible Fahrenheit ...,Plausibility,Flag and verify the original unit


WE HAVE SUCCESSFULLY DOCUMENTED ELEMENT 5, NOW ONTO PROBLEM #6: MULTIPLE INCONSISTENT HR VALUES FOR P001.

In [38]:
display(df[
    (df["patient_id"] == 1) &
    (df["metric"] == "heart_rate")
])

,patient_id,recorded_at,metric,value,unit,source


**ISSUE #6:** — FOUR INCONSISTENT HR VALUES FOR P001 EACH WITHIN 10 MINUTES OF EACH OTHER. *;00 AM, 8:05 AM, 8:10 AM and THE LAT TIME STAMP IS NaN.

* Row	Time	HR UNIT SOURCE
* 0	8:00 AM	72 bpm	EHR
* 1	8:05 AM	740 bpm	EHR
* 2	8:10 AM	68 BPM	Watch
* 3	Missing	70 bpm	EHR

ROW 1: 740 bpm IS PHYSIOLOGICALLY IMPLAUSIBLE FOR A HUMAN HR AND SUGGESTS A DATA-ENTRY OR MEASUREMENT ERROR. THE OTHER VALUES OF 72, 68, AND 70 ARE PLAUSIBLE. HR CAN CHANGE WITHIN MINUTES, SO DIFFERENT READINGS DON'T AUTOMATIVALLY INDICATE AN ERROR.

Field	Finding
Row	1
* FIELD FINDING: ISSUE- IMPOSSIBLY HIGH HR: 740 bpm
* DIMENSION: PLAUSIBIBILTY/ INACCURATE
* HANDLING:	FLAG AND EXCLUDE FROM THE HR SUMMARY FOR PENDING VERIFICATION.



In [39]:
issues.loc[len(issues)] = [
    1,
    "Implausibly high heart rate: 740 bpm",
    "Plausibility",
    "Flag and exclude from HR summary pending verification"
]

display(issues)

,Row,Quality issue,Dimension,Proposed handling
0,3,Missing recorded_at timestamp,Completeness,Flag for review
1,7,Impossible heart rate: -5.0 bpm,Plausibility,Flag and exclude from HR summary
2,12 and 13,Duplicate record: value 88,Concordance,Flag; remove one copy after verification
3,2 and 6,Inconsistent unit capitalization: BPM/bpm and ...,Correctness,Standardize unit labels after verification
4,10,98.6 recorded as Celsius; possible Fahrenheit ...,Plausibility,Flag and verify the original unit
5,1,Implausibly high heart rate: 740 bpm,Plausibility,Flag and exclude from HR summary pending verif...


STEP 4: CALCULATE DECSRIPTIVE STATISTICS FOR HR AFTER CLEANING: 🧬💚

SINCE WE HAVE ID'D PROBLEMATIC VALUES, WE WILL MAKE A CLEAN COPY OF MY FATA, EXCLUDE ANY INVALID HR MEASUREMENTS, AND RECORD THE MEAN, MEDIAN, MIN, MAX, AND SD.


In [40]:

# Make a copy so the original dataset stays unchanged
clean = df.copy()

# Standardize measurement names and units
clean["metric"] = clean["metric"].str.strip().str.lower()
clean["unit"] = clean["unit"].str.strip().str.lower()

# Remove exact duplicate records
clean = clean.drop_duplicates()

# Select heart-rate records only
hr = clean[clean["metric"] == "heart_rate"].copy()

# Convert values to numbers
hr["value"] = pd.to_numeric(hr["value"], errors="coerce")

# Keep bpm measurements with nonmissing, positive values
# Exclude the identified erroneous value of 740 bpm
hr_clean = hr[
    (hr["unit"] == "bpm") &
    (hr["value"] > 0) &
    (hr["value"] != 740)
].copy()

display(hr_clean)


,patient_id,recorded_at,metric,value,unit,source
0,P001,2026-09-01T08:00:00,heart_rate,72.0,bpm,ehr
2,P001,2026-09-01 08:10,heart_rate,68.0,bpm,watch
3,P001,NaN,heart_rate,70.0,bpm,ehr
8,P003,2026-09-01T10:00:00,heart_rate,58.0,bpm,watch
9,P003,2026-13-01T10:00:00,heart_rate,60.0,bpm,ehr
12,P005,2026-09-01T12:00:00,heart_rate,88.0,bpm,team_emr


STEP 2: CALCULATE STATISTICS...

In [41]:

print("Cleaned Heart Rate Summary")
print(hr_clean["value"].describe())


Cleaned Heart Rate Summary
count     6.000000
mean     69.333333
std      10.708252
min      58.000000
25%      62.000000
50%      69.000000
75%      71.500000
max      88.000000
Name: value, dtype: float64


**DOCUMENT CLEAN RULES:**
####STEP 4 — Heart Rate Cleaning and Descriptive Statistics

* The original dataset was preserved. For the heart-rate summary, measurement names and units were standardized, exact duplicate records were removed, and nonnumeric or missing heart-rate values were excluded. Nonpositive heart rates and the implausible 740 bpm measurement were excluded because they cannot be treated as reliable physiological observations without verification.

* Records which contained missing timestamps were flagged, though  retained in the descriptive summary when their heart-rate measurements were otherwise valid. The cleaned heart-rate measurements were then summarized using pandas.

CLEANING AND SUMMARIZING HR DATA...

In [43]:
print("Original HR records:", len(hr))
print("Cleaned HR records:", len(hr_clean))

display(hr_clean["value"].describe())

Original HR records: 9
Cleaned HR records: 6


,value
count,6.000000
mean,69.333333
std,10.708252
min,58.000000
25%,62.000000
50%,69.000000
75%,71.500000
max,88.000000


NOW WE HAVE OUR HR RECORD WITH INVALID MEASUREMENTS EXCLUDED...

NOW LETS DOCUMENT WHICH THOSE WERE...

In [44]:
display(hr_clean[["patient_id", "recorded_at", "value", "unit"]])

,patient_id,recorded_at,value,unit
0,P001,2026-09-01T08:00:00,72.0,bpm
2,P001,2026-09-01 08:10,68.0,bpm
3,P001,NaN,70.0,bpm
8,P003,2026-09-01T10:00:00,58.0,bpm
9,P003,2026-13-01T10:00:00,60.0,bpm
12,P005,2026-09-01T12:00:00,88.0,bpm


NOW LET'S CLEARLY DISPLAY THE STATISTICS...

In [45]:
print("Mean:", hr_clean["value"].mean())
print("Median:", hr_clean["value"].median())
print("Minimum:", hr_clean["value"].min())
print("Maximum:", hr_clean["value"].max())
print("Standard deviation:", hr_clean["value"].std())

Mean: 69.33333333333333
Median: 69.0
Minimum: 58.0
Maximum: 88.0
Standard deviation: 10.708252269472673


STEP 4: STATISTICAL INTERPRETATIONS...

### Task 4 — Descriptive Summary

After applied the documented cleaning rules, all HR measurements were summarized using pandas. The cleaned dataset had a mean of 69.3 (continued) bpm, a median of 69.0 bpm, and a range of 58.0 - 88.0 to bpm. The standard deviation was 10.708252269472673 bpm, describing the variability among the retained measurements. Implausible values are  excluded pending verification versus automatically correcting.

### Task 5 — Risks of Silent Auto-Cleaning

Silent auto-cleaning can negatively affect clinical decision-making since it changes/removes data without clearly documenting the reasons. Eg., auto-correcting a HR of fom 74 bpm to 740 bpm. This assumes a data-entry error without confirming the original measurement. Similarly, converting a recorded temperature of 98.6°C to 37°C assumes it was entered in Fahrenheit, which may conceal a documentation or equipment problem.

Automatically removing duplicate records could also eliminate legitimate repeated measurements if the system does not account for timestamps and data source types. It would be invalid and unethical to conclude that a patient's HR remained WNL if potentially abnormal measurements were silently removed without investigation.

These undocumented changes could lead clinicians to underestimate patient risk or make treatment decisions using false data. A safer approach is preserving the original records, flag questionable values for verification, document each cleaning decision, and involve clinical personnel when the correct interpretation is uncertain.